# 1. Data Processing

## Importing all the nesessary Libraries

In [84]:
# importing all the Necessary Libraries

import numpy as np
import pandas as pd
from pathlib import Path
import os


## Cleaning Steps

1. Add Patient ID column
2. Round numeric columns
3. Check the Datatypes
4. Convert Datatypes
5. Check the Outliers
    1. Glucose
    2. Calories
    3. Heart Rate

## Add Patient ID column to all the patient information files

In [85]:
def patientid(df,p_id):
    #Add Patient ID Column to the Patient information csv files
    # here we are paasing p_id (patient id value into the function so it know what to add
    df['patient_id'] = p_id
    return df

## Rounding all the numeric column values 

In [86]:
def rounding_numeric(df):
    # 1. Round numeric columns
    df = df.round(3)
    
    return df

In [87]:
df.dtypes

time                      datetime64[ns]
glucose                          float64
calories                         float64
heart_rate                       float64
steps                            float64
basal_rate                       float64
bolus_volume_delivered           float64
carb_input                       float64
patient_id                        object
dtype: object

In [88]:
def  fix_Dataypes(df): # 2. Convert time (Fixed the 'df1' typo here)
    
    df['time'] = pd.to_datetime(df['time'])
    
    return df
    

## Glucose Outliers

trying to remove biologically impoosible or indicating a sensor failure:\
**40mg/dl** is near the limit of survial; \
**500mg/dl** is a critical emergency.

In [89]:
def glucose_outliers(df):
        #remove impossible values for a living human
    # .Copy() without copy, if you delete the orinal big dataset, the filtered one might still be secretly tethered to it in memory.
    # but with adding copy() we can have a clean break. the filtered df stands on its own.
    df = df[(df['glucose']>= 40) & (df['glucose'] <= 500)].copy()
     # 6. Reset index
    df = df.reset_index(drop=True)
    return df

## Heart Rate Outliers

**Heart rate range**
**30-220 bpm** is a standard, resonable range for filtering impossible biological data in humans.

In [90]:
 def heart_rate_outliers(df): # Remove Heart Rate Outliers
    if 'heart_rate' in df.columns:
        # Remove impossible biological values 
        df = df[(df['heart_rate']>=30 ) & (df['heart_rate'] <= 220)].copy()
         # 6. Reset index
        df = df.reset_index(drop=True)
        return df

##  Uploading the Dataset and Cleaning

In [ ]:
# Create an empty list to store each cleaned DataFrame
all_patients_list = []
#Load your list of ID's from the csv file
patient_ids = pd.read_csv("T1DM_patient_sleep_demographics_with_race.csv")['Patient_ID'].tolist()

# Slice the list to take only the first 5 Patient IDs
first5patients = patient_ids[:5]

#Loop through only the ID's that acctually exist in your list
for i,p_id in enumerate(first5patients):
    input_file =f"{p_id}.csv"
    
    #Check if the file acctually exists in your computer before opening---------
    if os.path.exists(input_file):
        
        df = pd.read_csv(input_file,sep= ";")
        
    ### Add Patient ID column to all the patient information files--------------
        df = patientid(df,p_id)
     
    # CLEANING STEPS 
    # 1. rounding numeric 
        df = rounding_numeric(df)

    #2. Datypes checking and fixing 
        df = fix_Dataypes(df)

    #3. OUTLIERS
        df = glucose_outliers(df)
    
        df = heart_rate_outliers(df)

    # -------ADDING the CLEANED DF to our list
        all_patients_list.append(df)
        print(f"Processed {p_id})")
    
    # COMBINING all patient logs into one master table

        if all_patients_list:
            cleaned_data = pd.concat (all_patients_list, ignore_index = True)
        
    ##----------------SAVE TE CLEANED FILE----------------------------
            cleaned_data.to_csv("cleaned_file.csv", index = False)
        else:
            print("No files were found to process")



   